# 01 · 分词器与 BPE 直觉

模型吃的不是字符，而是 **token ID**。这一节回答三件事：

1. 文本怎么变成 ID、ID 怎么变回文本；
2. BPE 到底在切什么（为什么 "tokenization" 会被切成几块）；
3. 为什么词表大小必须和模型配置里的 `vocab_size` 严格一致。

用的是 `my_llm.tokenizer.build_tokenizer()`，底层是 tiktoken 的 `gpt2` 编码（词表 50257）。

In [1]:
from my_llm.tokenizer import build_tokenizer

tok = build_tokenizer("gpt2")
print("词表大小:", tok.n_vocab)
print("<|endoftext|> 的 ID:", tok.eot_token)

词表大小: 50257
<|endoftext|> 的 ID: 50256


## 1. 编解码往返

`encode` 返回 ID 列表，`decode` 是它的逆操作。注意一个坑：tiktoken 遇到 `<|endoftext|>`
这类特殊串时**默认直接抛 `ValueError`**（不是静默切碎）；要么显式 `allowed_special` 把它当
一个 token，要么自己先清理文本。所以 `TiktokenTokenizer.encode` 把
`allowed_special` 做成了显式关键字参数。

In [2]:
text = "Every effort moves you toward the goal."
ids = tok.encode(text)
print("ids:", ids)
print("decode:", tok.decode(ids))
assert tok.decode(ids) == text, "编解码必须能往返"

# 不显式允许时：特殊串会直接抛错，而不是被当成普通文本
try:
    tok.encode("hello <|endoftext|> world")
except ValueError as exc:
    print("未允许特殊 token -> ValueError:", str(exc).splitlines()[0])

print("显式允许:", tok.encode("hello <|endoftext|> world", allowed_special={"<|endoftext|>"}))

ids: [6109, 3626, 6100, 345, 3812, 262, 3061, 13]
decode: Every effort moves you toward the goal.
未允许特殊 token -> ValueError: Encountered text corresponding to disallowed special token '<|endoftext|>'.
显式允许: [31373, 220, 50256, 995]


## 2. BPE 直觉：常见词整块、生僻词切碎

BPE 的合并次数是学出来的：高频串会被合并成一个 token，低频串只能退回字节级。
下面把每个 token 单独解码出来看——注意有些块**不是完整单词**，甚至是不可打印的字节片段。

In [3]:
words = ["tokenization", "the", "GPT", "unbelievable", "hello"]
for w in words:
    ids_w = tok.encode(w)
    pieces = [tok.decode([i]) for i in ids_w]
    print(f"{w:<14} {len(ids_w)} token  {pieces}")

tokenization   2 token  ['token', 'ization']
the            1 token  ['the']
GPT            2 token  ['G', 'PT']
unbelievable   4 token  ['un', 'bel', 'iev', 'able']
hello          1 token  ['hello']


同一个意思，词频不同，token 数也不同——这直接决定了上下文能装多少内容，
也是为什么"同样的 1024 上下文"对不同语言/领域的有效长度并不一样。

In [4]:
samples = {
    "英文常见句": "The quick brown fox jumps over the lazy dog.",
    "中文": "从零实现一个语言模型。",
    "生僻英文": "Pneumonoultramicroscopicsilicovolcanoconiosis.",
}
for name, s in samples.items():
    ids_s = tok.encode(s)
    chars = len(s)
    print(f"{name:<8} {chars:>3} 字符 -> {len(ids_s):>3} token  ({chars / max(len(ids_s), 1):.2f} 字符/token)")

英文常见句     44 字符 ->  10 token  (4.40 字符/token)
中文        11 字符 ->  24 token  (0.46 字符/token)
生僻英文      46 字符 ->  16 token  (2.88 字符/token)


## 3. 字节层：为什么不会出现"未知词"

GPT-2 的 BPE 是**字节级**的，任意 UTF-8 字节都能被表示，所以没有 `<unk>`。
下面直接看 tiktoken 的字节表示（这一行用到的是 tiktoken 原生 API，
不在 `Tokenizer` 协议里，仅用于观察）。

In [5]:
for s in ["中", "文", "café"]:
    ids_s = tok.encode(s)
    print(f"{s!r:<8} ids={ids_s} bytes={tok.encoding.decode_tokens_bytes(ids_s)}")

'中'      ids=[40792] bytes=[b'\xe4\xb8\xad']
'文'      ids=[23877, 229] bytes=[b'\xe6\x96', b'\x87']
'café'   ids=[66, 1878, 2634] bytes=[b'c', b'af', b'\xc3\xa9']


## 小结

- `n_vocab` = 50257，必须等于 `GPTConfig.vocab_size`，否则 embedding 查表越界；
- 训练/推理两端的分词器必须是同一个，换成别的编码会让模型"看不懂"输入；
- 下一节看这些 ID 进去之后，attention 在做什么。